# RayNet · Colab runner

Run the cells **in order**. Read `docs/06_COMPUTE.md` first.

- Runtime → Change runtime type → **GPU** before you start.
- Secrets (key icon, left bar): `KAGGLE_USERNAME` + `KAGGLE_KEY`, or `KAGGLE_API_TOKEN`.
  Turn on "Notebook access".
- Everything that must survive a disconnect goes to Google Drive (`MyDrive/raynet`).
- Disconnect the runtime when you are done (Runtime → Disconnect and delete runtime).

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys, torch
print(sys.version)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| CPUs:", os.cpu_count())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU" 

## 2. Settings and Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/raynet"      # data zip, runs and checkpoints live here
COMMIT = "1d2db99"                           # our code, tested on the laptop
OFFICIAL_COMMIT = "fa3ef9b"                   # official baseline code
DATA = "/content/data/LunarRM"                # local disk copy (fast, for memmaps)
WORKERS = min(8, os.cpu_count() or 2)
SLUG = "the-first-lunar-pathloss-radio-map-prediction-challenge"
os.makedirs(f"{DRIVE}/data", exist_ok=True)
os.makedirs(f"{DRIVE}/runs", exist_ok=True)
os.makedirs(f"{DRIVE}/official_runs", exist_ok=True)
print("workers:", WORKERS)

## 3. Data to Drive (only once)
Downloads the competition zip from Kaggle with the token in Colab Secrets and stores it on
Drive. Skipped if `LunarRM.zip` is already on Drive.

In [ ]:
ZIP = f"{DRIVE}/data/LunarRM.zip"
if os.path.exists(ZIP):
    print("already on Drive:", ZIP, round(os.path.getsize(ZIP) / 1e9, 2), "GB")
else:
    from google.colab import userdata
    def secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None
    token, user, key = secret("KAGGLE_API_TOKEN"), secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
    elif user and key:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
    else:
        raise SystemExit("Add KAGGLE_API_TOKEN (or KAGGLE_USERNAME + KAGGLE_KEY) to Colab Secrets.")
    !pip install -q --upgrade kaggle
    !kaggle competitions download -c {SLUG} -p /content/dl
    !cp /content/dl/{SLUG}.zip "{ZIP}"
    print("saved", ZIP)

## 4. Our code at the tested commit

In [ ]:
!rm -rf /content/raynet-lunar
!git clone -q https://github.com/usarrahim/raynet-lunar.git /content/raynet-lunar
%cd /content/raynet-lunar
!git checkout -q {COMMIT}
!git log --oneline -1
!pip install -q -e .

## 5. Data: Drive → local disk (every session, a few minutes)

In [ ]:
import glob
if not glob.glob(f"{DATA}/**/metadata.json", recursive=True):
    !mkdir -p {DATA}
    !cp "{DRIVE}/data/LunarRM.zip" /content/data/LunarRM.zip
    !unzip -q -o /content/data/LunarRM.zip -d {DATA}
    !rm /content/data/LunarRM.zip
# the zip may contain a sub folder: use the folder that has metadata.json
DATA = os.path.dirname(sorted(glob.glob(f"{DATA}/**/metadata.json", recursive=True))[0])
print("DATA =", DATA)
!ls -la {DATA}

In [ ]:
# quick data check (about 1-2 minutes, no figure)
!python -m raynet.check_data --root {DATA} --no-fig | tail -15

## 6. P2.2: speed test of our model (about 10 minutes of GPU)
E3 at 5.8 GHz, 300 training steps for width 24 and 32. The `maps/s` numbers decide the
budget in `configs/base.yaml` (epochs so that one run takes about 1.5 h). Paste the two
`epoch 0` lines and the `peak_gpu_mem_gb` values to Claude Code.

In [ ]:
for w in (24, 32):
    !python -m raynet.train --config configs/e3_residual_58.yaml --data-root {DATA} --epochs 1 --limit-batches 300 --dev-max-batches 20 --width {w} --num-workers {WORKERS} --out /content/speed --run-name w{w} --log-every 0 | grep -E "params|epoch 0|peak_gpu|samples_per_s" 

## 7. P1.6: quick official baseline (RadioUNet, 5.8 GHz, 10 epochs, stage 1)
Through our wrapper: the dev terrains are removed from training and used as the monitor set.
The real `val` split is not used. Checkpoints go to Drive every epoch.

In [ ]:
!rm -rf /content/official
!git clone -q https://github.com/anderspearson206/lunar-radiomap-challenge.git /content/official
!cd /content/official && git checkout -q {OFFICIAL_COMMIT} && git log --oneline -1
OUT = f"{DRIVE}/official_runs"
RUN = "P16_58_s0_firstU" 

In [ ]:
# smoke test first (3 batches, about 1 minute)
!python scripts/run_official.py --official /content/official -- --data-root {DATA} --band 58 --loss masked --phase firstU --epochs 1 --limit-batches 3 --num-workers {WORKERS} --out /content/smoke --run-name smoke --panels 0

In [ ]:
# the real run (GPU hours!). If the runtime stops, run cells 1, 2, 4, 5 and this cell again:
# --resume auto continues from the last finished epoch.
!python scripts/run_official.py --official /content/official -- --data-root {DATA} --band 58 --loss masked --phase firstU --epochs 10 --seed 0 --num-workers {WORKERS} --out {OUT} --run-name {RUN} --resume auto

In [ ]:
# results on Drive (copy these to runs/ on the laptop; not the *.pt files)
!ls -la {OUT}
!cat {OUT}/{RUN}_history.csv